In [1]:
%pip install pandas transformers deepparse huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
    torch.set_default_device('cuda')
elif torch.xpu.is_available():
    print("XPU - available devices:")
    for i in range(torch.xpu.device_count()):
        print(f"  {i}: {torch.xpu.get_device_name(i)}")
    torch.set_default_device('xpu')
else: print("WARNING: Running on CPU")
print(f"Torch version: {torch.__version__}, Device: {torch.get_default_device()}")

XPU - available devices:
  0: Intel(R) Arc(TM) 130V GPU (16GB)
Torch version: 2.10.0+xpu, Device: xpu:0


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
import pandas as pd
from collections import OrderedDict

# Copilot suggestion, seems more efficient but needs testing
# def levenshtein(s1 : str, s2 : str) -> int:
#     if len(s1) < len(s2):
#         return levenshtein(s2, s1)

#     if len(s2) == 0:
#         return len(s1)

#     previous_row = range(len(s2) + 1)
#     for i, c1 in enumerate(s1):
#         current_row = [i + 1]
#         for j, c2 in enumerate(s2):
#             insertions = previous_row[j + 1] + 1
#             deletions = current_row[j] + 1
#             substitutions = previous_row[j] + (1 if c1 != c2 else 0)
#             current_row.append(min(insertions, deletions, substitutions))
#         previous_row = current_row

#     return previous_row[-1]

# Mahsa's code, more readable
def levenshtein(a: str, b: str) -> int:
    
    # If one of the strings is empty
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    # Create distance matrix (size: (len(a)+1) x (len(b)+1))
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    # Initialize first row/column
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    # Fill in matrix
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )
    distance = dp[-1][-1]

    return distance

def compare_preds(preds : pd.DataFrame, labels : pd.DataFrame, ignore_trash_columns = False):
    # Drop meta columns that may be included in the preds dataframe
    preds = preds.drop(columns=["fullResponse", "fullAddress", "parseError"], errors="ignore").astype(str)
    labels = labels.astype(str)

    tolerance_levels = 5
    correct_with_tol = [0,] * tolerance_levels
    total_preds = 0
    sum_levenshtein = 0
    sum_similarity = 0.0
    sum_levenshtein_match = 0
    sum_similarity_match = 0.0
    some_match_count = 0
    
    if not ignore_trash_columns:
        # labels that should not have been predicted at all
        trash_predictions = preds[[col for col in preds.columns if col not in labels.columns]].stack()
        total_preds += trash_predictions.notna().sum()
        sum_levenshtein += trash_predictions.dropna().astype(str).str.len().sum()
    for col in labels.columns:
        total_preds += len(labels)
        if col not in preds.columns:
            # all missing predictions are incorrect
            sum_levenshtein += labels[col].dropna().str.len().sum()
        else:
            strings_to_compare = pd.concat([preds[col].fillna(""), labels[col].fillna("")], axis=1)
            levenshtein_scores = strings_to_compare.apply(
                lambda row: levenshtein(row.iloc[0], row.iloc[1]), axis=1
            )
            levenshtein_bounds = strings_to_compare.apply(
                lambda row: max(len(row.iloc[0]), len(row.iloc[1])), axis=1
            )
            similarity = ((levenshtein_bounds - levenshtein_scores) / levenshtein_bounds).fillna(1.0) # nan => div by 0 => both are empty strings => similarity 1.0
            sum_levenshtein += levenshtein_scores.sum()
            sum_similarity += similarity.sum()
            sum_levenshtein_match += levenshtein_scores[similarity >= 0].sum()
            sum_similarity_match += similarity[similarity >= 0].sum()
            some_match_count += (similarity > 0).sum()
            for tol in range(tolerance_levels):
                correct_with_tol[tol] += (levenshtein_scores <= tol).sum()
    results = OrderedDict()
    results["accuracy"] = correct_with_tol[0] / total_preds
    for tol in range(1, tolerance_levels):
        results[f"accuracy_with_tol_{tol}"] = correct_with_tol[tol] / total_preds
    results["average_levenshtein"] = sum_levenshtein / total_preds
    results["average_similarity"] = sum_similarity / total_preds
    results["average_levenshtein_match"] = sum_levenshtein_match / some_match_count if some_match_count > 0 else 0.0
    results["average_similarity_match"] = sum_similarity_match / some_match_count if some_match_count > 0 else 0.0
    results["no_match_rate"] = 1.0 - (some_match_count / total_preds)
    return results

COLS_TO_PREDICT = [
    "HouseNumber",
    "StreetName",
    "City",
    "State",
    "Country"
]

EXCEPT_COUNTRY = COLS_TO_PREDICT[:-1]

In [5]:
import http.client
import json
import urllib.parse
import time

DEFAULT_LIBPOSTAL_LABEL_MAPPING = {
    "house_number": "HouseNumber",
    "road": "StreetName",
    "city": "City",
    "state": "State",
    "country": "Country",
    "postcode": "PostalCode"
}

class LibpostalClient:
    def __init__(self, url: str = "http://localhost:7272", label_mapping = DEFAULT_LIBPOSTAL_LABEL_MAPPING, start_container_if_unavailable: bool = True):
        self.url = url
        parsed_url = urllib.parse.urlparse(url)
        self.host = parsed_url.hostname
        self.port = parsed_url.port
        self.label_mapping = label_mapping or {}
        self.auto_started = False
        if start_container_if_unavailable:
            if not self.start_container_if_needed():
                raise ConnectionError(f"Could not connect to libpostal server at {self.url}, and failed to start docker container.")
    
    def _transform_results(self, parsed_addresses : list[list[list[str]]]) -> list[dict]:
        results = []
        for addr in parsed_addresses:
            result = {}
            for part, label in addr:
                if label in self.label_mapping:
                    label = self.label_mapping[label]
                if label in result:
                    result[label] += "___" + part
                else:
                    result[label] = part
            results.append(result)
        return results
    
    def _health_check(self) -> bool:
        conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
        try:
            conn.request("GET", "/health")
            response = conn.getresponse()
            conn.close()
            return response.status == 204
        except Exception:
            return False
    
    def _start_docker_container(self) -> bool:
        print("Attempting to start libpostal-server docker container...")
        print("This may take a long time on first run since the docker image needs to be built.")
        print("Note: By running this you are agreeing to conda TOS")
        import subprocess
        result = subprocess.run(["docker-compose", "-f", "docker-compose.yml", "up", "-d", "libpostal-server"], capture_output=True, text=True)
        if result.returncode != 0 or result.stderr != "":
            print(f"Failed to start libpostal-server docker container (exit code {result.returncode}):")
            print(result.stdout)
            print(result.stderr)
            return False
        for _ in range(10):
                time.sleep(1)
                if self._health_check():
                    print("Libpostal server is now available.")
                    self.auto_started = True
                    return True
        return False
    
    def start_container_if_needed(self):
        if not self._health_check():
            return self._start_docker_container()
        else: return True
            

    def parse_addresses(self, addresses: list[str]):
        def _request(addresses):
            conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
            headers = {'Content-Type': 'application/json'}
            if isinstance(addresses, str):
                addresses = [addresses]
            elif not isinstance(addresses, list):
                addresses = list(addresses)
            body = json.dumps(addresses)
            
            conn.request("GET", "/parser", body, headers)
            response = conn.getresponse()
            data = response.read()
            
            results = json.loads(data.decode('utf-8'))
            conn.close()

            return self._transform_results(results)
        try:
            return _request(addresses)
        except Exception as e:
            if self._health_check():
                raise e
            else:
                print(f"Libpostal server not reachable at {self.url}.")
                if not self._start_docker_container():
                    raise e
            return _request(addresses)
    
    def close(self):
        if self.auto_started:
            print("Stopping auto-started libpostal-server docker container...")
            import subprocess
            subprocess.run(["docker-compose", "-f", "docker-compose.yml", "down", "libpostal-server"])

In [6]:
import transformers
from transformers import pipeline

class LlamaAddressSegmentationModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=32):
        tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", padding_side='left')
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x : x)

    def parse_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt % {"address" : address}}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"]) for r in result]
        return responses

def extract_json_block(model_response : str):
    limit_chars = [('{', '}'), ('[', ']'), ('"', '"')]
    json_str = model_response
    parts = model_response.split("```")
    if len(parts) >= 2: # single code block or malformed code block
        json_str = parts[1]
    elif len(parts) >= 3:
        for part in parts:
            part = part.strip()
            if part.startswith("json") or any(part.startswith(lim[0]) and part.endswith(lim[1]) for lim in limit_chars):
                json_str = part
                break
    if json_str.startswith("json"):
        json_str = json_str[4:].strip()
    return json_str

In [7]:
import json

class JSONObjectParser:
    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            obj = json.loads(json_str)
            for k in obj.keys():
                assert k not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {k}"
                # It is plausible that the model just echoes back the full address in a field named "fullAddress"
                # But that would collide with our own field named "fullAddress" that we add later
                if k == "fullAddress":
                    obj["model-fullAddress"] = obj["fullAddress"] # avoid collision but keep the wrongfully generated field
            obj["fullResponse"] = response
            data = obj
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

class JSONTuplesParser:
    def __init__(self, ignore_other = True):
        self.ignore_other = ignore_other

    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            tuples = json.loads(json_str)
            data = {}
            for part, ptype in tuples:
                if isinstance(part, str) and part.strip() == "":
                    continue
                assert ptype not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {ptype}"
                if ptype == "fullAddress":
                    data["model-fullAddress"] = data["fullAddress"] # avoid collision but keep the wrongfully generated field
                else:
                    collision_value = data.get(ptype)
                    if collision_value is None:
                        data[ptype] = part
                    else:
                        data[ptype] = collision_value + "___" + part
            data["fullResponse"] = response
            if self.ignore_other:
                other_key = "Other"
                if isinstance(self.ignore_other, str): other_key = self.ignore_other
                if other_key in data:
                    del data[other_key]
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

In [8]:
from collections import OrderedDict
import json
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv")
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv")

EXAMPLES = [
    ("Berlin, Alexanderplatz 1, 10178", 
     OrderedDict([("City" , "Berlin"), ("StreetName", "Alexanderplatz"), ("HouseNumber", "1")])),
    ("Braunschweig, Uferstr. 25", # From BZK open training set
     OrderedDict([("City", "Braunschweig"), ("StreetName", "Uferstr."), ("HouseNumber", "25")])),
    ("808 Westend Avenue, New York 25, N.Y.", # From BZK open training set
        OrderedDict([("StreetName", "Westend Avenue"), ("HouseNumber", "808"), 
        ("City", "New York"), ("State", "N.Y.")])),
]

PROMPTS = [
    "Segment addresses into their components.\n"
    "Output only a JSON object with the following keys: " + ", ".join(COLS_TO_PREDICT) + ". "
    "Do not include fields that cannot be determined and do not try to guess their values. "
    "For example, if the address is simply \"Berlin\" then the field \"Country\" should be null. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps(ex)}\n```\n" for adr, ex in EXAMPLES) +
    "Now segment the following address:\n%(address)s",

    "Annotate address components with the respective types."
    "Consider the component types: " + ", ".join(COLS_TO_PREDICT + ["Other"]) + ". "
    "Not all addresses will contain all component types and you must not guess the missing ones. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Output only a JSON list of [component, type] tuples. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps([(v,k) for k,v in ex.items()])}\n```\n" for adr, ex in EXAMPLES) +
    "Now annotate the following address:\n%(address)s",
]

In [12]:

import time

model_configs = [
    # {
    #     "name" : "libpostal",
    #     "factory": lambda: LibpostalClient(),
    #     "cleanup": lambda client: client.close(),
    # },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt0",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt0",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt1",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt1",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    
]

preds_per_config = []

def eval(dataset, configs=model_configs):
    global preds_per_config
    model_results = []
    for config in configs:
        config_name = config["name"]
        print(f"Loading model {config_name}...")
        model = config["factory"]()
        print(f"Segmenting addresses...")
        start = time.monotonic()
        responses = model.parse_addresses(dataset["FullAddress"].tolist())
        end = time.monotonic()
        print("Parsing model responses...")
        preds_df = pd.DataFrame(responses)
        preds_per_config.append(preds_df)
        print("Computing metrics...")
        metrics = compare_preds(preds_df, dataset[COLS_TO_PREDICT])
        metrics["deltatime"] = end - start
        metrics["rate"] = len(dataset) / metrics["deltatime"]
        metrics["parseErrors"] = preds_df["parseError"].notna().sum() if "parseError" in preds_df.columns else 0
        metrics["errorRate"] = metrics["parseErrors"] / len(dataset)
        preds_df["config_name"] = config_name
        preds_df["FullAddress"] = dataset["FullAddress"]

        model_results.append(metrics)
        print(f"Results for model {config_name}: {metrics}")

        if "cleanup" in config:
            config["cleanup"](model)

    results_df = pd.DataFrame(model_results, index = [config["name"] for config in configs])
    return results_df

model_statistics = eval(bzkopen_val, model_configs)

model_statistics

Loading model Llama-3.2-1B-Instruct-prompt0...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Segmenting addresses...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Segmenting addresses...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Parsing model responses...
Computing metrics...
Results for model Llama-3.2-3B-Instruct-prompt1: OrderedDict({'accuracy': 0.8242424242424242, 'accuracy_with_tol_1': 0.8318181818181818, 'accuracy_with_tol_2': 0.8393939393939394, 'accuracy_with_tol_3': 0.8515151515151516, 'accuracy_with_tol_4': 0.8742424242424243, 'average_levenshtein': 1.2227272727272727, 'average_similarity': 0.8489359156595176, 'average_levenshtein_match': 1.3961937716262975, 'average_similarity_match': 0.9693731908914908, 'no_match_rate': 0.12424242424242427, 'deltatime': 270.73399999999674, 'rate': 0.48756343865196683, 'parseErrors': 5, 'errorRate': 0.03787878787878788})


,accuracy,accuracy_with_tol_1,accuracy_with_tol_2,accuracy_with_tol_3,accuracy_with_tol_4,average_levenshtein,average_similarity,average_levenshtein_match,average_similarity_match,no_match_rate,deltatime,rate,parseErrors,errorRate
Llama-3.2-1B-Instruct-prompt0,0.550681,0.590015,0.605144,0.627837,0.703480,2.930408,0.567927,4.901015,0.952791,0.403933,91.438,1.443601,3,0.022727
Llama-3.2-3B-Instruct-prompt0,0.791225,0.797277,0.806354,0.824508,0.856278,1.446293,0.815530,1.690941,0.957488,0.148260,311.828,0.423310,5,0.037879
Llama-3.2-1B-Instruct-prompt1,0.778281,0.788839,0.799397,0.820513,0.849170,1.631976,0.813821,1.867958,0.949935,0.143288,165.860,0.795852,16,0.121212
Llama-3.2-3B-Instruct-prompt1,0.824242,0.831818,0.839394,0.851515,0.874242,1.222727,0.848936,1.396194,0.969373,0.124242,270.734,0.487563,5,0.037879


In [10]:
preds_per_config_df = pd.concat(preds_per_config)
default_cols = ["config_name", "FullAddress"] + COLS_TO_PREDICT
new_order = default_cols + [col for col in preds_per_config_df.columns if col not in default_cols]
preds_per_config_df = preds_per_config_df[new_order]
preds_per_config_df

,config_name,FullAddress,HouseNumber,StreetName,City,State,Country,fullResponse,parseError,StreetType,...,ZipCode,District,Direction,OccupationalCode,Postal,PostalCode,City/Other,City/Town,City/State,State/Other
0,Llama-3.2-1B-Instruct-prompt0,"Regensburg, Königstr. 2/I",2,Königstr.,Regensburg,Bayern,NaN,Here is the JSON object with the specified fie...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-1B-Instruct-prompt0,Dortmund,0,Dortmund,Dortmund,NaN,NaN,Here is the JSON object with the requested fie...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-1B-Instruct-prompt0,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,Here's a Python function that uses regular exp...,Expecting value: line 1 column 1 (char 0),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.2-1B-Instruct-prompt0,"8 Burlington Road, Manchester 20/England.",8,Burlington Road,NaN,NaN,NaN,"{\n ""City"": null,\n ""StreetName"": ""Burlingto...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Llama-3.2-1B-Instruct-prompt0,Leer/Ostfriesland,NaN,Ostfriesland,NaN,NaN,NaN,Here is the JSON object with the requested fie...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,Sosnowice,NaN,Polen,"Here is the annotated address:\n\n```json\n[[""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,St.,NaN,N.Y.,USA,"Here is the annotated address:\n\n```json\n[[""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,NaN,Losone,NaN,NaN,"Based on the given address ""Losone CSR"", I wil...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CSR
130,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,"Since ""Rum."" does not contain any of the compo...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
preds_vs_trues = preds_per_config_df[default_cols].merge(bzkopen_val, on="FullAddress", suffixes=("_pred", "_true"), how="left")
preds_vs_trues

,config_name,FullAddress,HouseNumber_pred,StreetName_pred,City_pred,State_pred,Country_pred,filename,field,Unknown,...,HouseNumber_true,StreetName_true,Neighborhood,City_true,District,Region,State_true,Country_true,PostalCode,Other
0,Llama-3.2-1B-Instruct-prompt0,"Regensburg, Königstr. 2/I",2,Königstr.,Regensburg,Bayern,NaN,validation_0.jpg,ApplicantCurrentAddress,NaN,...,2/I,Königstr.,NaN,Regensburg,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-1B-Instruct-prompt0,Dortmund,0,Dortmund,Dortmund,NaN,NaN,validation_1.jpg,VictimBirthPlace,NaN,...,NaN,NaN,NaN,Dortmund,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-1B-Instruct-prompt0,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,validation_3.jpg,ApplicantBirthPlace,NaN,...,NaN,NaN,NaN,Jöhlingen,Krs. Durlach,NaN,Baden,NaN,NaN,NaN
3,Llama-3.2-1B-Instruct-prompt0,"8 Burlington Road, Manchester 20/England.",8,Burlington Road,NaN,NaN,NaN,validation_3.jpg,ApplicantCurrentAddress,NaN,...,8,Burlington Road,NaN,Manchester,NaN,NaN,NaN,England,NaN,NaN
4,Llama-3.2-1B-Instruct-prompt0,Leer/Ostfriesland,NaN,Ostfriesland,NaN,NaN,NaN,validation_4.jpg,ApplicantBirthPlace,NaN,...,NaN,NaN,NaN,Leer,NaN,NaN,NaN,NaN,NaN,Ostfriesland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,Sosnowice,NaN,Polen,validation_74.jpg,ApplicantBirthPlace,NaN,...,NaN,NaN,NaN,Sosnowice,NaN,NaN,NaN,Polen,NaN,NaN
572,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,St.,NaN,N.Y.,USA,validation_74.jpg,ApplicantCurrentAddress,NaN,...,NaN,NaN,NaN,St. Jackson Heights,NaN,NaN,N.Y.,USA,2114-79,NaN
573,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,NaN,Losone,NaN,NaN,validation_75.jpg,VictimBirthPlace,NaN,...,NaN,NaN,NaN,Losone,NaN,NaN,NaN,CSR,NaN,NaN
574,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,validation_76.jpg,ApplicantBirthPlace,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rum.,NaN,NaN
